# Web Scraping Book Data with BeautifulSoup — Practice Edition

This is the same scraping approach from `day18.ipynb` (fetch → parse → find containers → loop across pages → save), rebuilt against [books.toscrape.com](https://books.toscrape.com/) — a site built specifically for scraping practice, with no anti-bot protection, so this one actually runs end to end.

### A Quick Note on Scraping Responsibly

This site's whole purpose is to be scraped, so there's nothing to work around here — but it's still worth building the habit of checking `robots.txt` and adding delays between requests before scraping *any* site, including ones that do welcome it.

## 1. Imports

- `requests` — download each page's raw HTML
- `BeautifulSoup` — parse that HTML into something searchable
- `pandas` — assemble the results into a table
- `numpy` — `np.nan` for any field that fails to parse
- `time` — pause briefly between requests

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import time


## 2. Fetching and Parsing a Single Page

Every listing page follows the same URL pattern: `catalogue/page-{n}.html`. Page 1 has 20 books, and the whole catalogue spans 50 pages (1000 books total) — that's confirmed further down once the first page is parsed.

In [2]:
url = 'https://books.toscrape.com/catalogue/page-1.html'
webpage = requests.get(url).text
soup = BeautifulSoup(webpage, 'lxml')


`soup.prettify()` reformats the parsed tree with indentation — handy for exploring page structure while figuring out which tags/classes to target, noisy to leave in a final notebook (same as in the original day18 notebook).

In [3]:
# print(soup.prettify())


## 3. Finding the Container for Each Book

Just like AmbitionBox's `company-content-wrapper`, this site wraps each book listing in its own container element — here it's `<article class="product_pod">`. Grabbing all of them at once, and searching *within* each one individually, keeps each book's fields correctly paired together (rather than risking mismatched ordering from separately searching the whole page for all titles, then all prices, etc.).

In [4]:
books = soup.find_all('article', class_='product_pod')
len(books)   # should be 20 — one page's worth of listings


20

## 4. Exploring a Single Book's Fields

Before writing the extraction loop, it's worth pulling every field out of just the *first* container to confirm the right tags and classes:

- **Title** — sits in the `title` attribute of the `<a>` tag inside `<h3>` (the visible text is often truncated with `...`, so the attribute is the reliable source)
- **Price** — `<p class="price_color">`
- **Availability** — `<p class="instock availability">` (has extra whitespace, needs `.strip()`)
- **Star rating** — stored unusually: as a **CSS class** on `<p class="star-rating Three">`, not as text — the rating word (`One` through `Five`) is the *second* class in the list
- **Image URL** and **product page URL** — both relative paths, in `src`/`href` attributes

In [5]:
first = books[0]

title = first.h3.a['title']
price = first.find('p', class_='price_color').text
availability = first.find('p', class_='instock availability').text.strip()
rating = first.find('p', class_='star-rating')['class'][1]   # e.g. 'Three'
image_url = first.img['src']
product_url = first.h3.a['href']

print(title, price, availability, rating, image_url, product_url)


A Light in the Attic Â£51.77 In stock Three ../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg a-light-in-the-attic_1000/index.html


## 5. Extracting Every Field for a Whole Page

Same idea as day18's big extraction loop — for every container on the page, pull each field into its own list, wrapped in `try`/`except` so one book missing a field (e.g. no rating) doesn't crash the whole page. Then assemble everything into a single-page DataFrame.

In [6]:
def parse_page(soup_obj):
    """Given a parsed page, return a DataFrame of every book on it."""
    containers = soup_obj.find_all('article', class_='product_pod')

    title, price, availability, rating, image_url, product_url = [], [], [], [], [], []

    for c in containers:
        try:
            title.append(c.h3.a['title'])
        except (AttributeError, TypeError):
            title.append(np.nan)
        try:
            price.append(c.find('p', class_='price_color').text)
        except AttributeError:
            price.append(np.nan)
        try:
            availability.append(c.find('p', class_='instock availability').text.strip())
        except AttributeError:
            availability.append(np.nan)
        try:
            rating.append(c.find('p', class_='star-rating')['class'][1])
        except (AttributeError, IndexError, TypeError):
            rating.append(np.nan)
        try:
            image_url.append(c.img['src'])
        except (AttributeError, TypeError):
            image_url.append(np.nan)
        try:
            product_url.append(c.h3.a['href'])
        except (AttributeError, TypeError):
            product_url.append(np.nan)

    return pd.DataFrame({
        'title': title, 'price': price, 'availability': availability,
        'rating': rating, 'image_url': image_url, 'product_url': product_url,
    })

page_df = parse_page(soup)
page_df.head()


,title,price,availability,rating,image_url,product_url
0,A Light in the Attic,Â£51.77,In stock,Three,../media/cache/2c/da/2cdad67c44b002e7ead0cc356...,a-light-in-the-attic_1000/index.html
1,Tipping the Velvet,Â£53.74,In stock,One,../media/cache/26/0c/260c6ae16bce31c8f8c95dadd...,tipping-the-velvet_999/index.html
2,Soumission,Â£50.10,In stock,One,../media/cache/3e/ef/3eef99c9d9adef34639f51066...,soumission_998/index.html
3,Sharp Objects,Â£47.82,In stock,Four,../media/cache/32/51/3251cf3a3412f53f339e42cac...,sharp-objects_997/index.html
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,../media/cache/be/a5/bea5697f2534a2f86a3ef27b5...,sapiens-a-brief-history-of-humankind_996/index...


## 6. Looping Across All 50 Pages

Same pagination pattern as day18's `for j in range(1, 1001)`, but with the fixes already covered: collect each page's DataFrame in a list and `pd.concat()` once at the end (not the removed `df.append()`), check `status_code` before parsing, and pause briefly between requests.

In [7]:
all_pages = []

for page_num in range(1, 51):
    page_url = f'https://books.toscrape.com/catalogue/page-{page_num}.html'
    resp = requests.get(page_url)

    if resp.status_code != 200:
        print(f'Page {page_num} failed with status {resp.status_code}, skipping')
        continue

    page_soup = BeautifulSoup(resp.text, 'lxml')
    all_pages.append(parse_page(page_soup))

    time.sleep(0.1)   # good habit even on a site that welcomes scraping

books_df = pd.concat(all_pages, ignore_index=True)
books_df.shape


(1000, 6)

In [8]:
books_df.sample(5)


,title,price,availability,rating,image_url,product_url
777,Something Blue (Darcy & Rachel #2),Â£54.62,In stock,One,../media/cache/72/f5/72f5ed312bc82afa386c9cd48...,something-blue-darcy-rachel-2_223/index.html
223,Walt Disney's Alice in Wonderland,Â£12.96,In stock,Five,../media/cache/28/50/2850439c2ba103fb69dba9cd2...,walt-disneys-alice-in-wonderland_777/index.html
507,The Constant Princess (The Tudor Court #1),Â£16.62,In stock,Three,../media/cache/fc/80/fc80b999ff4b8ef24b7071f62...,the-constant-princess-the-tudor-court-1_493/in...
318,Crazy Love: Overwhelmed by a Relentless God,Â£47.72,In stock,Two,../media/cache/03/f1/03f1e337afadba35687672b56...,crazy-love-overwhelmed-by-a-relentless-god_682...
399,Off the Hook (Fishing for Trouble #1),Â£47.67,In stock,Three,../media/cache/1d/78/1d78fe226e1adb9cb591fa21f...,off-the-hook-fishing-for-trouble-1_601/index.html


## 7. Saving the Result — `to_csv()`

(This step was missing entirely from the original day18 notebook — the scraped `final` DataFrame there was never written to disk. Not repeating that here.)

In [9]:
books_df.to_csv('books.csv', index=False)


## 8. Checking Data Quality

Since missing fields were filled with `np.nan`, and the same book could theoretically appear twice if a page were fetched twice, checking for both is worth doing right after any scrape.

In [10]:
print(books_df.isnull().sum())
print('Duplicate rows:', books_df.duplicated().sum())


title           0
price           0
availability    0
rating          0
image_url       0
product_url     0
dtype: int64
Duplicate rows: 0


## 9. Bonus: Cleaning the Scraped Columns

Raw scraped data is almost always messy in predictable ways — worth cleaning up right after scraping, before any analysis:
- `price` is a string like `'£51.77'` — strip the currency symbol and convert to a proper number
- `rating` is a word (`'Three'`) because of how the site encodes it as a CSS class — map it to an actual integer

In [11]:
books_df['price'] = books_df['price'].str.replace('£', '', regex=False).astype(float)

rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
books_df['rating'] = books_df['rating'].map(rating_map)

books_df.head()


ValueError: could not convert string to float: 'Â51.77'

## Summary

| Step | Method |
|---|---|
| Fetch a page | `requests.get(url).text` |
| Parse it | `BeautifulSoup(html, 'lxml')` |
| Find every listing on a page | `soup.find_all('article', class_='product_pod')` |
| Extract a field, safely | `try: ... except AttributeError: append(np.nan)` |
| Collect many pages | list of DataFrames + `pd.concat(list, ignore_index=True)` |
| Be a polite scraper | `time.sleep()` between requests, check `status_code` |
| Save | `df.to_csv('file.csv', index=False)` |
| Check data quality | `.isnull().sum()`, `.duplicated().sum()` |
| Clean scraped strings | `.str.replace()` + `.astype(float)`, `.map()` for coded values |